# Local Whisper no Google Colab

Gera legendas `.srt` em português a partir de vídeos em qualquer idioma no Google Drive, em duas etapas: o [faster-whisper](https://github.com/SYSTRAN/faster-whisper) transcreve no idioma original e o [NLLB](https://huggingface.co/facebook/nllb-200-distilled-600M) traduz para português, ambos na GPU.

**Antes de rodar:** `Ambiente de execução` → `Alterar tipo de ambiente de execução` → **T4 GPU**.

Fluxo:
1. Verificar GPU
2. Instalar dependências
3. Montar o Google Drive
4. Configurar pasta de entrada, profundidade da varredura e modelos
5. Definir as funções auxiliares
6. Carregar o Whisper e o tradutor
7. Transcrever, traduzir e salvar as legendas ao lado de cada vídeo

Para `video.mkv` em japonês, são gerados `video.ja.srt` (original) e `video.srt` (tradução).

> O idioma é detectado por vídeo, não por trecho. O NLLB tem licença CC-BY-NC, que não permite uso comercial.

## 1. Verificar GPU

Se `nvidia-smi` falhar, o ambiente não está com GPU — troque o runtime antes de continuar.

In [ ]:
!nvidia-smi

## 2. Instalar dependências

`av<14` reproduz a pin do projeto local. `transformers` e `sentencepiece` rodam o tradutor. O `ctranslate2` precisa de CUDA 12 e cuDNN 9, que o runtime do Colab pode não trazer. Por isso a célula instala essas bibliotecas via pip e as registra no sistema com `ldconfig`.

In [ ]:
!pip install -q "faster-whisper>=1.2.1" "av<14" transformers sentencepiece nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"

import site
import subprocess
from pathlib import Path

# LD_LIBRARY_PATH só vale no início do processo; o ldconfig funciona com o kernel já rodando.
cuda_lib_dirs = set()
for packages_dir in site.getsitepackages():
    for lib_name in ("libcublas.so.12", "libcudnn.so.9"):
        for lib in Path(packages_dir).glob(f"nvidia/**/{lib_name}"):
            cuda_lib_dirs.add(str(lib.parent))

if not cuda_lib_dirs:
    raise RuntimeError("libcublas.so.12 e libcudnn.so.9 não encontradas após o pip install")

Path("/etc/ld.so.conf.d/cuda12-pip.conf").write_text("\n".join(sorted(cuda_lib_dirs)) + "\n")
subprocess.run(["ldconfig"], check=True)

ldconfig_output = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True).stdout
for lib_name in ("libcublas.so.12", "libcudnn.so.9"):
    status = "ok" if lib_name in ldconfig_output else "NÃO ENCONTRADA"
    print(f"[*] {lib_name}: {status}")

## 3. Montar o Google Drive

Os vídeos devem estar em alguma pasta do seu Drive. Cada legenda será salva com o mesmo nome do vídeo (`video.mp4` → `video.srt`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Configuração

- **INPUT_PATH**: pasta ou arquivo de vídeo dentro do Drive.
- **SCAN_DEPTH**: quantos níveis de subpastas varrer abaixo de `INPUT_PATH`. `0` = só a própria pasta; `1` = filmes (`filmes/Nome do Filme/filme.mkv`); `2` = séries (`series/Nome da Série/Season 01/ep.mkv`).
- **OUTPUT_PATH**: pasta de saída, que espelha as subpastas de `INPUT_PATH` (vazio = salvar ao lado do vídeo).
- **MODEL_SIZE**: `large-v3` aproveita o T4 e dá o melhor resultado.
- **SOURCE_LANGUAGE**: idioma do áudio (`ja`, `en`...). Vazio = detectar automaticamente. Preencha se a detecção errar, ou ao rodar uma pasta de um idioma só.
- **TRANSLATION_MODEL**: `600M` é o mais leve; `1.3B` tende a traduzir melhor, mas ocupa mais VRAM e é mais lento.
- **FORCE**: `True` reprocessa mesmo que já exista uma legenda.

In [ ]:
INPUT_PATH = "/content/drive/MyDrive/videos"  # @param {type:"string"}
SCAN_DEPTH = 2  # @param {type:"integer"}
OUTPUT_PATH = ""  # @param {type:"string"}
MODEL_SIZE = "large-v3"  # @param ["tiny", "base", "small", "medium", "large-v3"]
SOURCE_LANGUAGE = ""  # @param {type:"string"}
TRANSLATION_MODEL = "facebook/nllb-200-distilled-600M"  # @param ["facebook/nllb-200-distilled-600M", "facebook/nllb-200-distilled-1.3B"]
BEAM_SIZE = 5  # @param {type:"integer"}
FORCE = False  # @param {type:"boolean"}

## 5. Funções auxiliares

Adaptação do pacote `local_whisper` (`files.py`, `srt.py`, `transcriber.py`) para rodar em um único notebook, mais a etapa de tradução.

In [ ]:
from __future__ import annotations

import os
import re
import time
from dataclasses import replace
from pathlib import Path

import torch
from tqdm.auto import tqdm

VIDEO_EXTENSIONS = {".mp4", ".mkv", ".avi", ".mov", ".webm", ".flv", ".wmv", ".m4v", ".ts"}

SUBTITLE_EXTENSIONS = {
    ".srt", ".vtt", ".ass", ".ssa", ".sub", ".sbv",
    ".ttml", ".dfxp", ".smi", ".stl", ".usf", ".idx",
}

TARGET_LANGUAGE = "pt"
TARGET_NLLB_CODE = "por_Latn"
TRANSLATION_BATCH_SIZE = 32
LANGUAGE_DETECTION_SEGMENTS = 5

NLLB_LANGUAGE_CODES = {
    "ar": "arb_Arab", "de": "deu_Latn", "en": "eng_Latn", "es": "spa_Latn",
    "fr": "fra_Latn", "hi": "hin_Deva", "id": "ind_Latn", "it": "ita_Latn",
    "ja": "jpn_Jpan", "ko": "kor_Hang", "nl": "nld_Latn", "pl": "pol_Latn",
    "ru": "rus_Cyrl", "th": "tha_Thai", "tr": "tur_Latn", "uk": "ukr_Cyrl",
    "vi": "vie_Latn", "zh": "zho_Hans",
}

SENTENCE_BOUNDARY = re.compile(r"(?<=[.!?…])\s+|(?<=[。！？])\s*")


def collect_videos(path: str, depth: int) -> list[Path]:
    root_path = Path(path)
    if root_path.is_file():
        return [root_path] if root_path.suffix.lower() in VIDEO_EXTENSIONS else []

    videos: list[Path] = []
    for current_dir, subdirs, filenames in os.walk(root_path):
        current_path = Path(current_dir)
        level = len(current_path.relative_to(root_path).parts)
        if level >= depth:
            subdirs.clear()
        for filename in filenames:
            file_path = current_path / filename
            if file_path.suffix.lower() in VIDEO_EXTENSIONS:
                videos.append(file_path)
    return sorted(videos)


def existing_subtitle(video: Path, extra_dir: Path | None = None) -> Path | None:
    dirs = [video.parent]
    if extra_dir and extra_dir != video.parent:
        dirs.append(extra_dir)
    for d in dirs:
        for ext in SUBTITLE_EXTENSIONS:
            candidate = d / (video.stem + ext)
            if candidate.exists():
                return candidate
    return None


def format_timestamp(seconds: float) -> str:
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    ms = int((seconds - int(seconds)) * 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def segments_to_srt(segments: list) -> str:
    lines: list[str] = []
    for i, seg in enumerate(segments, start=1):
        lines.append(str(i))
        lines.append(f"{format_timestamp(seg.start)} --> {format_timestamp(seg.end)}")
        lines.append(seg.text.strip())
        lines.append("")
    return "\n".join(lines)


def write_srt(dest: Path, segments: list) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(segments_to_srt(segments), encoding="utf-8")


def split_sentences(text: str) -> list[str]:
    parts = (part.strip() for part in SENTENCE_BOUNDARY.split(text))
    return [part for part in parts if part]


def translate_texts(translator, tokenizer, texts: list[str], source_language: str) -> list[str]:
    source_code = NLLB_LANGUAGE_CODES.get(source_language)
    if source_code is None:
        raise ValueError(f"idioma '{source_language}' sem código NLLB em NLLB_LANGUAGE_CODES")

    tokenizer.src_lang = source_code
    target_token_id = tokenizer.convert_tokens_to_ids(TARGET_NLLB_CODE)

    translated: list[str] = []
    for i in range(0, len(texts), TRANSLATION_BATCH_SIZE):
        batch = texts[i:i + TRANSLATION_BATCH_SIZE]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True)
        inputs = inputs.to(translator.device)
        with torch.inference_mode():
            output_tokens = translator.generate(**inputs, forced_bos_token_id=target_token_id)
        translated.extend(tokenizer.batch_decode(output_tokens, skip_special_tokens=True))
    return translated


def translate_segments(translator, tokenizer, segments: list, source_language: str) -> list:
    # O NLLB descarta frases quando recebe várias de uma vez, então cada uma é traduzida sozinha.
    sentences_by_segment = [split_sentences(seg.text) for seg in segments]
    all_sentences = [s for sentences in sentences_by_segment for s in sentences]
    translated = translate_texts(translator, tokenizer, all_sentences, source_language)

    result = []
    position = 0
    for seg, sentences in zip(segments, sentences_by_segment):
        next_position = position + len(sentences)
        text = " ".join(translated[position:next_position])
        result.append(replace(seg, text=text))
        position = next_position
    return result


def transcribe_file(model, translator, tokenizer, video: Path, output_dir: Path | None,
                    index: int = 0, total: int = 0,
                    source_language: str = "", beam_size: int = 5) -> Path:
    base_dir = output_dir or video.parent
    dest = base_dir / (video.stem + ".srt")
    batch = f" ({index}/{total})" if total else ""
    print(f"  → {video.name}{batch}")

    start = time.perf_counter()
    segments_gen, info = model.transcribe(
        str(video),
        language=source_language or None,
        language_detection_segments=LANGUAGE_DETECTION_SEGMENTS,
        beam_size=beam_size,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=500),
    )
    print(f"    idioma: {info.language} (confiança {info.language_probability:.2f})")

    duration = info.duration
    collected = []
    bar = tqdm(
        total=int(duration),
        unit="s",
        bar_format="    {l_bar}{bar}| {n_fmt}/{total_fmt}s [{elapsed}<{remaining}]",
        leave=False,
    )
    prev_end = 0.0
    for seg in segments_gen:
        collected.append(seg)
        delta = seg.end - prev_end
        if delta > 0:
            bar.update(int(delta))
            prev_end = seg.end
    bar.update(bar.total - bar.n)
    bar.close()

    elapsed = time.perf_counter() - start

    if not collected:
        print(f"    ⚠ nenhuma fala detectada ({elapsed:.1f}s)")
        return dest

    if info.language == TARGET_LANGUAGE:
        write_srt(dest, collected)
        print(f"    ✓ {len(collected)} segmentos — {elapsed:.1f}s → {dest.name}")
        return dest

    # Salvo antes de traduzir para não perder a transcrição se a tradução falhar.
    original_dest = base_dir / f"{video.stem}.{info.language}.srt"
    write_srt(original_dest, collected)
    print(f"    ✓ {len(collected)} segmentos — {elapsed:.1f}s → {original_dest.name}")

    translate_start = time.perf_counter()
    translated = translate_segments(translator, tokenizer, collected, info.language)
    write_srt(dest, translated)
    translate_elapsed = time.perf_counter() - translate_start
    print(f"    ✓ traduzido — {translate_elapsed:.1f}s → {dest.name}")
    return dest

## 6. Carregar os modelos

Whisper em `float16` na GPU (rápido e enxuto na VRAM do T4) ou `int8` como fallback de CPU, e depois o tradutor NLLB. O primeiro uso baixa os dois modelos.

In [ ]:
import ctranslate2
import torch
from faster_whisper import WhisperModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

if ctranslate2.get_cuda_device_count() > 0:
    device, compute_type = "cuda", "float16"
else:
    device, compute_type = "cpu", "int8"
    print("[!] GPU não detectada — rodando em CPU (vai ser lento). Troque o runtime para T4.")

model = WhisperModel(MODEL_SIZE, device=device, compute_type=compute_type)
print(f"[*] Whisper: {MODEL_SIZE} | Dispositivo: {device} ({compute_type})")

translator_device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(TRANSLATION_MODEL)
translator = AutoModelForSeq2SeqLM.from_pretrained(TRANSLATION_MODEL).to(translator_device)
print(f"[*] Tradutor: {TRANSLATION_MODEL} | Dispositivo: {translator_device}")

## 7. Transcrever e traduzir

Pula vídeos que já têm legenda (a menos que `FORCE=True`). Erros em um arquivo não interrompem o lote. Se a tradução falhar, a transcrição original continua salva em `video.<idioma>.srt`.

In [ ]:
videos = collect_videos(INPUT_PATH, SCAN_DEPTH)
if not videos:
    raise SystemExit(f"Nenhum vídeo encontrado em {INPUT_PATH} (SCAN_DEPTH={SCAN_DEPTH})")

input_path = Path(INPUT_PATH)
input_root = input_path if input_path.is_dir() else input_path.parent
output_root = Path(OUTPUT_PATH) if OUTPUT_PATH else None


def output_dir_for(video: Path) -> Path | None:
    if output_root is None:
        return None
    return output_root / video.parent.relative_to(input_root)


if not FORCE:
    filtered = []
    for v in videos:
        sub = existing_subtitle(v, output_dir_for(v))
        if sub:
            print(f"[=] pulando {v.name} — já tem legenda ({sub.name})")
        else:
            filtered.append(v)
    videos = filtered

if not videos:
    print("\n[✓] Nada a fazer — todos os vídeos já têm legenda")
else:
    print(f"[*] {len(videos)} vídeo(s) para transcrever\n")
    for i, video in enumerate(videos, start=1):
        try:
            transcribe_file(model, translator, tokenizer, video, output_dir_for(video),
                            index=i, total=len(videos),
                            source_language=SOURCE_LANGUAGE, beam_size=BEAM_SIZE)
        except Exception as e:
            print(f"    ✗ erro em {video.name}: {e}")
    print("\n[✓] Concluído")